# 📦 ITEC Product Re-categorization — Full Pipeline

> จัดหมวดสินค้า ITEC ใหม่ทั้งหมด แก้ปัญหา category เดิม **key ผิด / ซ้ำซ้อน**
> Feature: `ItemName` · Target: หมวดที่คุณตั้งเอง · Backbone: LLM embedding (ไม่ fine-tune)

---

## 🗺️ ภาพรวมขั้นตอนสร้างโมเดล (อ่านก่อนรัน)

```
ItemName ดิบ
   │
① Normalize          ทำความสะอาดข้อความ                         [Cell 3]
   │
② สร้าง Flags         คำใน ItemName → IS_ flags (keyword)         [Cell 4-5] ✏️
   │                  เช่น "iphone" → IS_iPhone=1
   │
③ จัดหมวด (Label)     รวม flags → หมวดใหญ่ = y                    [Cell 6] ✏️
   │                  เช่น IS_Case → "อุปกรณ์เสริม"
   │
④ Embedding          LLM อ่าน ItemName → เวกเตอร์ (แช่แข็ง)       [Cell 8]
   │
⑤ เทรน Classifier     LogisticRegression: เวกเตอร์ → หมวด         [Cell 9-10]
   │                  ★ ส่วนที่ "เรียน" จริง มีแค่ตรงนี้
   │
⑥ ประเมิน            Accuracy + Macro-F1                        [Cell 11]
   │
⑦ ทำ label สะอาด      จับ item ที่ key ผิด → แก้ → retrain         [Cell 12]
   │
⑧ (option) Fine-tune  ปรับ weights LLM — ทำท้ายสุด ถ้ายังไม่พอ    [Cell 13]
```

## 🎚️ 3 ระดับที่คุณตั้งเองได้

| ระดับ | คือ | แก้ที่ |
|---|---|---|
| 1. **Flag** (ชื่ออังกฤษ) | มี flag อะไรบ้าง | Cell 4 ✏️ |
| 2. **คำใน flag** (ไทย/eng) | keyword ที่ทำให้ติด flag | Cell 4 ✏️ |
| 3. **จัดหมวด flag** (label) | flag ไหนรวมเป็นหมวดใหญ่ | Cell 6 ✏️ |

## ⚠️ ลำดับความคุ้ม (ทำตามนี้)

```
1. embedding + classifier (baseline)  ← ทำก่อนเสมอ
2. ทำ label สะอาด + เพิ่มข้อมูล        ← คุ้มสุด
3. fine-tune                          ← ท้ายสุด · ดีขึ้น "ต่อเมื่อ label สะอาดก่อน"
```

> 🔑 fine-tune บน label ที่ยัง noisy = เรียนของผิดแม่นขึ้น (แย่ลง) → ต้องทำ label สะอาดก่อน

## 📌 หมายเหตุ
- นี่**ไม่ใช่ fine-tune** — LLM แช่แข็ง เรียนแค่ LogisticRegression
- 2 โมเดลเทียบ: M1 random-drop (predict ItemName ล้วน) · M2 full (predict ครบ)
- CPU ช้า → ใช้ subset / MiniLM · GPU → bge-m3 (ดูตาราง performance ใน Obsidian)
- ✏️ = cell ที่ต้องแก้ · cell อื่นรันผ่านได้เลย

## 1 · Setup

In [ ]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"       # กันค้างตอนโหลด (Windows)
import re, numpy as np, pandas as pd, joblib
print("ready · pandas", pd.__version__)

## 2 · ✏️ โหลดข้อมูล — ต้องมี ItemName, CategoryName, SubCategoryName, Brand

In [ ]:
# df = pd.read_csv("dim_item_itec.csv")
for col in ["CategoryName","SubCategoryName","Brand"]:
    if col not in df.columns: df[col] = ""
print("rows:", len(df)); df.head()

## 3 · Normalize

In [ ]:
def normalize(s: pd.Series) -> pd.Series:
    return (s.fillna("").astype(str).str.lower()
             .str.replace(r"[^\w\s/&+.-]", " ", regex=True)
             .str.replace(r"\s+", " ", regex=True).str.strip())

df["search_text"] = (normalize(df.ItemName)+" | "+normalize(df.CategoryName)+" | "+normalize(df.SubCategoryName))
df[["ItemName","search_text"]].head()

## 4 · ✏️✏️ FLAG_RULES — ตั้ง Flag (อังกฤษ) + คำ (ไทย/eng)

`whole_word=True` สำหรับคำสั้น กัน false positive

In [ ]:
FLAG_RULES = {
    "IS_iPhone":     {"keywords": ["iphone","ไอโฟน"], "whole_word": False},
    "IS_Smartphone": {"keywords": ["smartphone","smart phone","มือถือ","โทรศัพท์"], "whole_word": False},
    "IS_Galaxy":     {"keywords": ["galaxy"], "whole_word": False},
    "IS_iPad":       {"keywords": ["ipad","ไอแพด"], "whole_word": False},
    "IS_Tablet":     {"keywords": ["tablet","แท็บเล็ต"], "whole_word": False},
    "IS_Notebook":   {"keywords": ["notebook","laptop","โน้ตบุ๊ก"], "whole_word": False},
    "IS_Macbook":    {"keywords": ["macbook"], "whole_word": False},
    "IS_PC":         {"keywords": ["pc","คอมพิวเตอร์"], "whole_word": True},
    "IS_iMac":       {"keywords": ["imac"], "whole_word": False},
    "IS_GraphicCard":{"keywords": ["graphic card","การ์ดจอ","vga","gpu"], "whole_word": False},
    "IS_RAM":        {"keywords": ["ram","แรม"], "whole_word": True},
    "IS_CPU":        {"keywords": ["cpu","ซีพียู"], "whole_word": True},
    "IS_Harddisk":   {"keywords": ["harddisk","hdd","ssd","ฮาร์ดดิสก์","external"], "whole_word": False},
    "IS_Mainboard":  {"keywords": ["mainboard","motherboard","เมนบอร์ด"], "whole_word": False},
    "IS_AirPod":     {"keywords": ["airpod"], "whole_word": False},
    "IS_Headset":    {"keywords": ["headset","headphone","หูฟัง"], "whole_word": False},
    "IS_Speaker":    {"keywords": ["speaker","ลำโพง"], "whole_word": False},
    "IS_Watch":      {"keywords": ["watch","นาฬิกา"], "whole_word": True},
    "IS_Mouse":      {"keywords": ["mouse","เมาส์"], "whole_word": True},
    "IS_Keyboard":   {"keywords": ["keyboard","คีย์บอร์ด"], "whole_word": False},
    "IS_Camera":     {"keywords": ["camera","กล้อง"], "whole_word": False},
    "IS_Monitor":    {"keywords": ["monitor","จอคอม"], "whole_word": False},
    "IS_Adapter":    {"keywords": ["adapter","อะแดปเตอร์"], "whole_word": False},
    "IS_Charger":    {"keywords": ["charger","ที่ชาร์จ"], "whole_word": False},
    "IS_PowerBank":  {"keywords": ["power bank","powerbank","พาวเวอร์แบงก์"], "whole_word": False},
    "IS_Cable":      {"keywords": ["cable","สายชาร์จ","สายไฟ"], "whole_word": True},
    "IS_Case":       {"keywords": ["case","เคส"], "whole_word": True},
    "IS_Cover":      {"keywords": ["cover","ฝาหลัง"], "whole_word": True},
    "IS_Protect":    {"keywords": ["screen protector","tempered","ฟิล์ม"], "whole_word": False},
    "IS_Strap":      {"keywords": ["strap","สายคล้อง"], "whole_word": False},
    "IS_Stand":      {"keywords": ["stand","ขาตั้ง"], "whole_word": True},
    "IS_AppleCare":  {"keywords": ["applecare","apple care","care+"], "whole_word": False},
    "IS_Insurance":  {"keywords": ["insurance","ประกัน"], "whole_word": False},
    "IS_Software":   {"keywords": ["software","license","office"], "whole_word": False},
    "IS_Playstation":{"keywords": ["playstation","ps5","ps4"], "whole_word": False},
    "IS_Nintendo":   {"keywords": ["nintendo","switch"], "whole_word": False},
    # flag พิเศษ (multi-label เสริม)
    "IS_Promotion":  {"keywords": ["promotion","โปรโมชั่น","ของแถม","bundle"], "whole_word": False},
    "IS_Demo":       {"keywords": ["demo","tester","ตัวโชว์"], "whole_word": True},
    "IS_Gaming":     {"keywords": ["gaming","เกมมิ่ง","rog","predator"], "whole_word": False},
}
print(f"flags: {len(FLAG_RULES)}")

## 5 · สร้าง flags (ไม่ต้องแก้)

In [ ]:
def build_flags(search_text, rules):
    out = pd.DataFrame(index=search_text.index)
    for flag,cfg in rules.items():
        kws=cfg.get("keywords",[])
        if not kws: out[flag]=0; continue
        ww=cfg.get("whole_word",False)
        pat="|".join((rf"\b{re.escape(w.lower())}\b" if ww else re.escape(w.lower())) for w in kws)
        out[flag]=search_text.str.contains(pat,regex=True).astype("int8")
    return out

flags = build_flags(df.search_text, FLAG_RULES)
df = pd.concat([df.drop(columns=[c for c in flags.columns if c in df.columns],errors="ignore"), flags], axis=1)
print(flags.sum().sort_values(ascending=False).head(15))

## 6 · ✏️✏️ MY_TAXONOMY — จัดกลุ่ม flag → หมวด (label)

**ลำดับสำคัญ! ตัวแรกที่ตรงชนะ** — Accessory บนสุด แก้ "iPhone Case → มือถือ" 

In [ ]:
def _has(d,*cols):
    cols=[c for c in cols if c in d.columns]
    return d[cols].any(axis=1) if cols else pd.Series(False,index=d.index)

MY_TAXONOMY = [
    ("อุปกรณ์เสริม",   lambda d:_has(d,"IS_Case","IS_Cover","IS_Protect","IS_Strap","IS_Stand",
                                      "IS_Charger","IS_Adapter","IS_Cable","IS_PowerBank")),
    ("ประกัน&บริการ",  lambda d:_has(d,"IS_AppleCare","IS_Insurance")),
    ("มือถือ&แท็บเล็ต",lambda d:_has(d,"IS_iPhone","IS_Smartphone","IS_Galaxy","IS_iPad","IS_Tablet")),
    ("เสียง&สวมใส่",   lambda d:_has(d,"IS_AirPod","IS_Headset","IS_Speaker","IS_Watch")),
    ("ชิ้นส่วนคอม",    lambda d:_has(d,"IS_GraphicCard","IS_RAM","IS_CPU","IS_Harddisk","IS_Mainboard")),
    ("คอมพิวเตอร์",    lambda d:_has(d,"IS_Notebook","IS_Macbook","IS_PC","IS_iMac")),
    ("อุปกรณ์ต่อพ่วง", lambda d:_has(d,"IS_Mouse","IS_Keyboard","IS_Monitor","IS_Camera")),
    ("เกม",           lambda d:_has(d,"IS_Playstation","IS_Nintendo")),
    ("ซอฟต์แวร์",      lambda d:_has(d,"IS_Software")),
]
df["MyCategory"] = np.select([f(df).values for _,f in MY_TAXONOMY],
                             [l for l,_ in MY_TAXONOMY], default="อื่นๆ")
print(df.MyCategory.value_counts())
# ⚠️ "อื่นๆ" ใหญ่เกิน = flags ไม่พอ → เพิ่มใน Cell 4

## 7 · เตรียม X, y

In [ ]:
vc=df.MyCategory.value_counts()
data=df[df.MyCategory.isin(vc[vc>=5].index)].reset_index(drop=True)
y=data.MyCategory.values
print(f"{len(data):,} แถว · {pd.Series(y).nunique()} หมวด")
# ⚡ CPU ช้า? → data=data.sample(5000,random_state=42).reset_index(drop=True); y=data.MyCategory.values

## 8 · โหลด backbone (auto GPU/CPU)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
device="cuda" if torch.cuda.is_available() else "cpu"
backbone="BAAI/bge-m3" if device=="cuda" else "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder=SentenceTransformer(backbone,device=device)
print(f"backbone={backbone} · device={device}")

def make_text(item,cat="",sub="",brand="",drop_extra=False):
    if drop_extra: cat=sub=brand=""
    p=[normalize(pd.Series([item]))[0]]
    if str(cat).strip():   p.append("หมวด: "+normalize(pd.Series([cat]))[0])
    if str(sub).strip():   p.append("ซับ: "+normalize(pd.Series([sub]))[0])
    if str(brand).strip(): p.append("brand: "+normalize(pd.Series([brand]))[0])
    return " | ".join(p)

## 9 · โมเดล 1 — Random-drop (predict ItemName ล้วน)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score,classification_report
rng=np.random.default_rng(42)
t1=[make_text(r.ItemName,r.CategoryName,r.SubCategoryName,r.Brand,drop_extra=(rng.random()<0.5)) for r in data.itertuples()]
X1=embedder.encode(t1,batch_size=64,show_progress_bar=True,normalize_embeddings=True)
Xtr,Xte,ytr,yte,itr,ite=train_test_split(X1,y,np.arange(len(y)),test_size=0.2,random_state=42,stratify=y)
clf1=LogisticRegression(max_iter=2000,C=10,class_weight="balanced",n_jobs=-1).fit(Xtr,ytr)
tt=[make_text(data.iloc[i].ItemName,drop_extra=True) for i in ite]
p1=clf1.predict(embedder.encode(tt,normalize_embeddings=True))
acc1,f1a=accuracy_score(yte,p1),f1_score(yte,p1,average="macro")
print(f"[M1] Acc={acc1:.4f} MacroF1={f1a:.4f}")

## 10 · โมเดล 2 — Full (predict ครบ)

In [ ]:
t2=[make_text(r.ItemName,r.CategoryName,r.SubCategoryName,r.Brand) for r in data.itertuples()]
X2=embedder.encode(t2,batch_size=64,show_progress_bar=True,normalize_embeddings=True)
clf2=LogisticRegression(max_iter=2000,C=10,class_weight="balanced",n_jobs=-1).fit(X2[itr],ytr)
p2=clf2.predict(X2[ite])
acc2,f2a=accuracy_score(yte,p2),f1_score(yte,p2,average="macro")
print(f"[M2] Acc={acc2:.4f} MacroF1={f2a:.4f}")

## 11 · เปรียบเทียบ + บันทึก

In [ ]:
print(pd.DataFrame({"Model":["1.Random-drop(ItemName)","2.Full(ครบ)"],
                    "Accuracy":[acc1,acc2],"Macro-F1":[f1a,f2a]}).to_string(index=False))
joblib.dump({"backbone":backbone,"clf":clf1,"taxonomy":[l for l,_ in MY_TAXONOMY],"flag_rules":FLAG_RULES},"model1_itemname.joblib")
joblib.dump({"backbone":backbone,"clf":clf2},"model2_full.joblib")
print("saved 2 models")

## 12 · ⭐ ทำ label สะอาด — จับ item ที่ key ผิด

ML ทายต่างจาก label เดิม + มั่นใจสูง = น่าจะ key ผิด → export ให้คนตรวจ → แก้ → retrain
(ขั้นนี้คุ้มกว่า fine-tune — ทำก่อน)

In [ ]:
proba=clf1.predict_proba(X1); 
data["Cat_pred"]=clf1.classes_[proba.argmax(1)]; data["conf"]=proba.max(1)
suspect=data[(data.Cat_pred!=data.MyCategory)&(data.conf>=0.80)].sort_values("conf",ascending=False)
print(f"น่าสงสัยว่า key ผิด: {len(suspect):,}")
suspect[["ItemName","CategoryName","MyCategory","Cat_pred","conf"]].to_csv("suspect.csv",index=False,encoding="utf-8-sig")
suspect[["ItemName","MyCategory","Cat_pred","conf"]].head(20)

## 13 · (Optional) Fine-tune — ทำท้ายสุด ถ้า baseline ยังไม่พอ

⚠️ ต้อง GPU + label สะอาด (ทำ Cell 12 ก่อน) · fine-tune บน label noisy = แย่ลง
วิธี contrastive: สอน bge-m3 ให้สินค้าหมวดเดียวกัน embedding ใกล้กัน

In [ ]:
# ต้อง GPU — ตัวอย่าง contrastive fine-tune
# from sentence_transformers import SentenceTransformer, InputExample, losses
# from torch.utils.data import DataLoader
# model = SentenceTransformer("BAAI/bge-m3", device="cuda")
# examples = []  # สร้างคู่ (item1,item2,label) จาก MyCategory ที่สะอาดแล้ว
# for cat, grp in data.groupby("MyCategory"):
#     items = grp.ItemName.tolist()
#     # คู่บวก (หมวดเดียวกัน) + คู่ลบ (คนละหมวด) ...
# loader = DataLoader(examples, batch_size=16, shuffle=True)
# loss = losses.CosineSimilarityLoss(model)
# model.fit(train_objectives=[(loader,loss)], epochs=1, warmup_steps=100)
# model.save("bge-m3-itec-finetuned")
# → แล้วกลับไปรัน Cell 8-11 ใหม่ด้วย backbone นี้ เทียบ accuracy
print("fine-tune: ทำเมื่อ baseline ไม่พอ + label สะอาดแล้ว (ดู Obsidian: ITEC Category Re-grouping)")